# Sudanese Regional LLM - Kaggle QLoRA Fine-Tuning (Background Run)

This notebook fine-tunes **Qwen/Qwen2.5-7B-Instruct** on Sudanese dialectal datasets using **QLoRA** (4-bit quantization).
When executed via **Save & Run All (Commit)**, Kaggle will run this in the background for up to 12 hours even if you close your browser.

📌 **Important Note on Kaggle Internet Access**:
- Kaggle requires phone verification to enable external internet access in notebooks.
- Ensure **Settings -> Internet -> On** is enabled in the Kaggle UI notebook editor.

In [ ]:
# Pre-flight check & Install dependencies
import socket
import sys
import subprocess

def check_internet():
    try:
        socket.create_connection(("8.8.8.8", 53), timeout=3)
        print("🌐 Internet connection is ACTIVE.")
        return True
    except OSError:
        print("⚠️ WARNING: Internet is OFF or restricted on this Kaggle session.")
        print("👉 To enable: Go to Kaggle Notebook Settings panel -> Toggle 'Internet' to ON.")
        return False

internet_available = check_internet()

if internet_available:
    print("📦 Upgrading bitsandbytes, trl, peft...")
    # Upgrade bitsandbytes, trl, peft without upgrading torch/transformers to prevent CUDA/torchvision conflicts
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "bitsandbytes", "trl", "peft"])
else:
    print("ℹ️ Skipping pip install due to missing internet connection. Using pre-installed Kaggle packages.")

In [ ]:
import os
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Fallback import for SFTTrainer
try:
    from trl import SFTTrainer
    HAS_TRL = True
    print("✅ Imported SFTTrainer from TRL")
except ImportError:
    HAS_TRL = False
    print("ℹ️ TRL package not found. Will fallback to standard Transformers Trainer.")

# Authenticate with Hugging Face if secrets are available
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    print("✅ Successfully logged in to Hugging Face")
except Exception as e:
    print(f"ℹ️ Hugging Face login skipped or token not found: {e}")

In [ ]:
# Dataset: Sudanese Dialect Knowledge Base
training_data = [
    {"instruction": "ما هي اللهجة المستخدمة في الخرطوم وما أبرز الملامح؟", "response": "اللهجة الخرطومية هي اللهجة الحضرية المركزية في السودان، وتستخدم كلمات مثل 'زول' (شخص)، 'كيفنك' (كيف حالك)، و'حبابك' (أهلاً بك)."},
    {"instruction": "ترجم بلهجة دارفور: كيف حالك يا صديقي؟", "response": "اللهجة الدارفورية: 'أبوي كيفنك والمال عافية؟' أو 'يا زول كيفك وكيف أمورك؟'"},
    {"instruction": "ما هو المثل الشعبي الشائع في كردفان للتضامن؟", "response": "في كردفان يُقال: 'الصلح خير والإيد الواحدة ما بتصفق' للحث على النفير والتعاون في حصاد الصمغ العربي والزراعة."},
    {"instruction": "ما معنى كلمة 'شنقلي' أو 'حبابك عشام' في الشمالية؟", "response": "في الشمالية والمنطقة النوبية، 'حبابك عشرة بلا كشرة' تعني الترحيب الحار والكرام الحفي بالأضياف."},
    {"instruction": "كيف يرحب أهل الشرق (البجا) بالضيف؟", "response": "في شرق السودان يُقال 'عافيات' و'حبابك' مع تقديم القهوة الجبنة كرمز للإكرام والضيافة البجاوية."}
]

# Format for Qwen Chat Template
def format_prompts(batch):
    formatted = []
    for inst, resp in zip(batch['instruction'], batch['response']):
        text = f"<|im_start|>user\n{inst}<|im_end|>\n<|im_start|>assistant\n{resp}<|im_end|>"
        formatted.append(text)
    return {"text": formatted}

dataset = Dataset.from_list(training_data).map(format_prompts, batched=True)
print(f"✅ Dataset prepared: {len(dataset)} examples")

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# Load Model (4-bit QLoRA with float16 fallback)
if torch.cuda.is_available():
    try:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True
        )
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True
        )
        model = prepare_model_for_kbit_training(model)
        print("✅ Loaded model using 4-bit BitsAndBytes QLoRA.")
    except Exception as e:
        print(f"⚠️ 4-bit Quantization failed ({e}). Loading in fp16 precision...")
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True
        )
else:
    print("⚠️ GPU not detected. Loading model in CPU mode...")
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32, device_map="cpu", trust_remote_code=True)

In [ ]:
# LoRA Adapter Configuration
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# Training Arguments
training_args = TrainingArguments(
    output_dir="./sudanese-llm-lora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=30,
    learning_rate=2e-4,
    fp16=torch.cuda.is_available(),
    logging_steps=5,
    save_strategy="no",
    push_to_hub=False
)

if HAS_TRL:
    trainer = SFTTrainer(
        model=model,
        train_dataset=dataset,
        dataset_text_field="text",
        max_seq_length=512,
        args=training_args
    )
else:
    def tokenize_func(examples):
        return tokenizer(examples['text'], truncation=True, max_length=512)
    tokenized_ds = dataset.map(tokenize_func, batched=True)
    trainer = Trainer(
        model=model,
        train_dataset=tokenized_ds,
        data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
        args=training_args
    )

print("🚀 Starting Fine-Tuning...")
trainer.train()
print("✅ Training Complete!")

In [ ]:
# Save trained LoRA adapters locally
model.save_pretrained("./sudanese-llm-lora-final")
tokenizer.save_pretrained("./sudanese-llm-lora-final")

# Push to Hugging Face Hub if token available
try:
    model.push_to_hub("goro806/sudanese-llm-lora")
    tokenizer.push_to_hub("goro806/sudanese-llm-lora")
    print("🎉 Successfully pushed fine-tuned LoRA adapters to Hugging Face Hub!")
except Exception as e:
    print(f"ℹ️ Could not push to HF Hub: {e}")